# Reproduce the quick-profile figures

Run `spde-pf run --config configs/quick.yaml --output runs/quick --workers 4` first. This notebook only loads generated outputs; it does not duplicate simulation or filtering logic.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

run = Path("../runs/quick")
truth = np.load(run / "truth.npz")
observations = np.load(run / "observations.npz")
filters = {r: np.load(run / f"filter_{r}.npz") for r in (32, 16, 8, 4, 2)}
with (run / "metrics.csv").open() as handle:
    metrics = list(csv.DictReader(handle))
metrics

In [ ]:
step = min(100, truth["activator"].shape[0] - 1)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), constrained_layout=True)
axes[0].imshow(truth["activator"][step], origin="lower", cmap="viridis")
axes[1].imshow(observations["counts_32"][step - 1], origin="lower", cmap="magma")
axes[2].imshow(filters[32]["mean_u"][step], origin="lower", cmap="viridis")
for ax, title in zip(axes, ("True activator", "Poisson counts", "Posterior mean"), strict=True):
    ax.set_title(title)
    ax.set_axis_off()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
for resolution, result in filters.items():
    ax.plot(result["rmse"], label=f"{resolution}×{resolution}")
ax.plot(filters[32]["open_loop_rmse"], "k--", label="open-loop baseline")
ax.set(xlabel="observation step", ylabel="activator RMSE")
ax.legend(ncol=2)
plt.show()